# Risk Stratified Table

**This notebook generates a Table 1 for patients with advanced head and neck cancer receiving first-line pembrolizumab plus chemotherapy versus pembrolizumab, showing covariate differences stratified by the threshold effect.** 

In [1]:
import numpy as np
import pandas as pd

from tableone import TableOne

## Import data

In [2]:
treatment_df = pd.read_csv('../outputs/pembrochemo_pembro_index.csv')

In [3]:
treatment_df.sample(3)

,PatientID,LineName,StartDate
834,F0F3B95AF22FC,pembro,2020-04-14
340,FB68BC331785B,pembro_platinum,2022-01-27
525,F2C924CE2EA68,pembro_platinum,2023-06-07


In [4]:
treatment_df.shape

(1854, 3)

In [5]:
treatment_df['treatment'] = (treatment_df['LineName'] == 'pembro_platinum').astype(int)

In [6]:
dtype_map = pd.read_csv('../outputs/pembrochemo_pembro_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/pembrochemo_pembro_features_df.csv', dtype = dtype_map)

In [7]:
features_df.shape

(1736, 162)

In [8]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [9]:
surv_pred_df.head(3)

,PatientID,psurv_180_calibrated
0,F33D856BE4FC6,0.936744
1,FEBCC4F65CC4C,0.973762
2,F2C55497C35F5,0.000000


In [10]:
df = pd.merge(features_df, treatment_df, on = 'PatientID', how = 'left')

In [11]:
df.shape

(1736, 165)

In [12]:
df = pd.merge(df, surv_pred_df, on = 'PatientID', how = 'left')

In [13]:
df.shape

(1736, 166)

In [14]:
#for ASCO abstract
from lifelines import KaplanMeierFitter

# Reverse KM: treat deaths as censored, censoring as events
kmf = KaplanMeierFitter()
kmf.fit(durations=df['duration'], event_observed=(df['event'] == 0)) 
median_followup = kmf.median_survival_time_

print(f'median age: {df.age.median()}')
print(f'percent male: {df.sex_male.sum()/len(df)}')
print(f'median follow-up: {median_followup/30}')

median age: 68.0
percent male: 0.7845622119815668
median follow-up: 24.2


## Binning relevant covariates

In [14]:
df['weight_loss_5p'] = np.where(df['percent_change_weight'] <= -5, 1, 0)

In [15]:
df['creatinine_2'] = np.where(df['creatinine'] > 2, 1, 0)

In [16]:
df['hemoglobin_9'] = np.where(df['hemoglobin'] < 9, 1, 0)

In [17]:
df['ast_60'] = np.where(df['ast'] > 60, 1, 0)

In [18]:
df['albumin_3'] = np.where(df['albumin'] < 30, 1, 0)

In [19]:
df['alp_200'] = np.where(df['alp'] > 200, 1, 0)

In [20]:
df['above_threshold'] = np.where(df['psurv_180_calibrated'] >= 0.63, 1, 0)

## Creating table 

In [21]:
table1 = TableOne(
    data = df, 
    columns = ['age', 'GroupStage_mod', 'SmokingStatus', 'HPVStatus_mod', 'received_surgery', 'PDL1_status', 'ecog_index', 'weight_loss_5p', 'creatinine_2', 'ast_60', 'hemoglobin_9', 'albumin_3', 'alp_200', 'thoracic_met', 'liver_met', 'brain_met', 'bone_met'],
    categorical = ['GroupStage_mod', 'HPVStatus_mod', 'SmokingStatus', 'received_surgery', 'PDL1_status', 'ecog_index', 'weight_loss_5p', 'creatinine_2', 'ast_60', 'hemoglobin_9', 'albumin_3', 'alp_200', 'thoracic_met', 'liver_met', 'brain_met', 'bone_met'],
    continuous = ['age'],
    nonnormal = ['age'],
    groupby = 'above_threshold',
    pval = True)

In [22]:
table1

Grouped by above_threshold                                                              
                                                    Missing           Overall                 0                 1 P-Value
n                                                                        1736               502              1234        
age, median [Q1,Q3]                                       0  68.0 [61.0,75.0]  69.0 [61.0,78.0]  67.0 [60.2,73.0]  <0.001
GroupStage_mod, n (%)   1.0                                         138 (7.9)          31 (6.2)         107 (8.7)   0.220
                        2.0                                        205 (11.8)         52 (10.4)        153 (12.4)        
                        3.0                                        272 (15.7)         76 (15.1)        196 (15.9)        
                        4.0                                        943 (54.3)        288 (57.4)        655 (53.1)        
                        5.0                                        178 (10.3)         55 (11.0)        123 (10.0)        
SmokingStatus, n (%)    0                                          426 (24.5)        112 (22.3)        314 (25.4)   0.189
                        1                                         1310 (75.5)        390 (77.7)        920 (74.6)        
HPVStatus_mod, n (%)    negative                                   640 (36.9)        229 (45.6)        411 (33.3)  <0.001
                        positive                                   669 (38.5)         98 (19.5)        571 (46.3)        
                        unknown                                    427 (24.6)        175 (34.9)        252 (20.4)        
received_surgery, n (%) 0                                         1148 (66.1)        288 (57.4)        860 (69.7)  <0.001
                        1                                          588 (33.9)        214 (42.6)        374 (30.3)        
PDL1_status, n (%)      positive                                   311 (17.9)         85 (16.9)        226 (18.3)   0.541
                        unknown                                   1425 (82.1)        417 (83.1)       1008 (81.7)        
ecog_index, n (%)       0.0                                        414 (23.8)         51 (10.2)        363 (29.4)  <0.001
                        1.0                                       1084 (62.4)        301 (60.0)        783 (63.5)        
                        2.0                                        194 (11.2)        115 (22.9)          79 (6.4)        
                        3.0                                          40 (2.3)          32 (6.4)           8 (0.6)        
                        4.0                                           4 (0.2)           3 (0.6)           1 (0.1)        
weight_loss_5p, n (%)   0                                         1324 (76.3)        282 (56.2)       1042 (84.4)  <0.001
                        1                                          412 (23.7)        220 (43.8)        192 (15.6)        
creatinine_2, n (%)     0                                         1717 (98.9)        492 (98.0)       1225 (99.3)   0.042
                        1                                            19 (1.1)          10 (2.0)           9 (0.7)        
ast_60, n (%)           0                                         1689 (97.3)        473 (94.2)       1216 (98.5)  <0.001
                        1                                            47 (2.7)          29 (5.8)          18 (1.5)        
hemoglobin_9, n (%)     0                                         1644 (94.7)        435 (86.7)       1209 (98.0)  <0.001
                        1                                            92 (5.3)         67 (13.3)          25 (2.0)        
albumin_3, n (%)        0                                         1679 (96.7)        456 (90.8)       1223 (99.1)  <0.001
                        1                                            57 (3.3)          46 (9.2)          11 (0.9)        
alp_200

In [23]:
table1.to_csv('../outputs/risk_stratified_table1.csv', header = True)

In [24]:
df.groupby('above_threshold')['ecog_index_na'].sum()

above_threshold
0    122
1    292
Name: ecog_index_na, dtype: int64